# H infinity condition for stabilization of a discrete time system

In [19]:
import numpy as np
import cvxpy as cp
import control as ct

Matrizes do sistema

In [20]:
A  = np.array([
    [1, 0.035],
    [0, -0.13]
])
Bu = np.array([
    [-0.002],
    [-0.11]
])
Bw = np.array([
    [0.1],
    [0.1]
])
C  = np.array([
    [1, 0]
])
# C = np.eye(A.shape[0])
Du = np.zeros(shape=(C.shape[0], Bu.shape[1]))
Dw = np.zeros(shape=(C.shape[0], Bw.shape[1]))

In [21]:
nx = A.shape[0]
nu = Bu.shape[1]
nc = C.shape[0]

eps = 10e-19 # 

gamma = cp.Variable()

Z = cp.Variable((nx, nx), symmetric=True)
W = cp.Variable((nu, nx))
P = cp.Variable((nx, nx), symmetric=True)

X32 = -np.eye(nc, dtype=float) # C@C.T

constrains = []
constrains += [ P >> eps ]

# B11 = -2*Z - P - A@Z - Bu@W - Z@A.T - W.T@Bu.T
# B12 = Z - Z@A.T - W.T@Bu.T
# B13 = -Z@C.T@X32.T - W.T@Du.T@X32.T
# B14 = -Bw

# B22 = -P + 4*Z
# B23 = np.zeros(shape=(nx, nc))
# B24 = -Bw

# B33 = -np.eye(nc, dtype=float)
# B34 = -X32@Dw

# B44 = -np.eye(1)*gamma

B11 = -2*Z + P - A@Z - Bu@W - Z@A.T - W.T@Bu.T
B12 = Z + Z@A.T + W.T@Bu.T
B13 = -Z@C.T@X32.T - W.T@Du.T@X32.T
B14 = -Bw

B22 = P - 4*Z
B23 = np.zeros(shape=(nx, nc))
B24 = Bw

B33 = np.eye(nc, dtype=float) + X32 + X32.T
B34 = -X32@Dw

B44 = -np.eye(1)*gamma

block = cp.bmat([
    [B11  , B12  , B13  , B14],
    [B12.T, B22  , B23  , B24],
    [B13.T, B23.T, B33  , B34],
    [B14.T, B24.T, B34.T, B44]
])
constrains += [ block << -eps]

prob = cp.Problem(cp.Minimize(gamma), constraints=constrains)
prob.solve(solver=cp.MOSEK, verbose=True)

                                     CVXPY                                     
                                     v1.5.3                                    
(CVXPY) Jan 06 08:43:50 PM: Your problem has 11 variables, 40 constraints, and 0 parameters.
(CVXPY) Jan 06 08:43:50 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jan 06 08:43:50 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jan 06 08:43:50 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jan 06 08:43:50 PM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Jan 06 08:43:50 PM: Compiling problem (target solver=MOSEK).
(C

0.0018682891092935462

In [22]:
X = np.linalg.inv(Z.value)
K = W.value@X
K

array([[1411.95578318,  -93.34032796]])

In [23]:
all(np.linalg.eig(P.value).eigenvalues > 0)

True

In [24]:
gamma.value

array(0.00186829)